# Basic Revision

In [2]:
!pip install --quiet pydantic

# 1. Basic Pydantic Model

In [3]:
from pydantic import BaseModel

In [6]:
class Patient(BaseModel):
  name: str
  age: int

def insert_data(data: Patient):
  print(data.name)
  print(data.age)

patient_info = {'name':'Sayam', 'age': 26}
patient_object1 = Patient(**patient_info) # Create Object and validate

insert_data(patient_object1) # pass the object in any function

Sayam
26


# 2. With Lists, Dicts and Optional

In [18]:
from typing import List, Dict, Optional, Annotated
from pydantic import AnyUrl, EmailStr, Field

class Patient(BaseModel):
  name: Annotated[str, Field(description='name of the person')]
  age: int
  allergies: Optional[List[str]] = None # Can also be optional, thus None is assigned
  contact_Details: Dict[str, str]
  website: AnyUrl


patient_info = {'name':'Sayam', 'age': 26, 'contact_Details': {'phone':'8575761143'}, 'website': 'https://s'}

patient1 = Patient(**patient_info)
patient1.contact_Details

{'phone': '8575761143'}

# 3. Field Validator

In [24]:
from pydantic import field_validator

class Patient(BaseModel):
  name: Annotated[str, Field(description='name of the person')]

  @field_validator('name')
  @classmethod
  def validate_name(cls, value):
    # Just to show an example, can be done in field too if logic is simple

    if len(value) < 3:
      raise TypeError('Length should be more than 3')


patient_info = {'name': 'sam'}
patient_obj_1 = Patient(**patient_info)

# 4. Model Validator

In [29]:
from pydantic import model_validator

class Patient(BaseModel):
  name: str = Field(..., description='name of the person')
  age: Annotated[int, Field(description='age of the person')]
  contact: Dict[str, str]

  @model_validator(mode='after')
  def validate_age(cls, model):
    if model.age > 60 and 'emergency' not in model.contact:
      raise TypeError('Include emrgency contact details')

    return model

patient_info = {'name': 'sam', 'age': 65, 'contact':{'ph': '99', 'emergency':'5'}}
p = Patient(**patient_info)

/tmp/ipython-input-3180238945.py:8: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.12/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode='after')


# 5. Nested Model

In [32]:
class Address(BaseModel):
  city: str
  state: str

class Patient(BaseModel):
  name: str = Field(description='name of the person')
  address: Address

address_info = {'city': 'boston', 'state': 'MA'}
address_1 = Address(**address_info)
patient_info = {'name': 'sam', 'address': address_1}


# 6. Serialization

In [36]:
class Patient(BaseModel):
  name: str
  age: int


patient_info = {'name':'Sayam', 'age': 26}
patient_object1 = Patient(**patient_info)
data = patient_object1.model_dump()

In [37]:
print(data)

{'name': 'Sayam', 'age': 26}


# Practice Questions

In [40]:
# Level 1: Basic Models
# Question 1.1: Simple User Model
# Create a Pydantic model for this data:

pythondata = {
    "username": "sayam_kumar",
    "email": "sayam@example.com",
    "age": 25,
    "is_active": True
}

# Write the model and parse the data.

In [43]:
from pandas.core.frame import DataFrame
from pydantic import BaseModel, Field
from typing import List, Dict, Annotated

class User(BaseModel):
  username: str
  email: str
  age: int
  is_active: bool


obj = User(**pythondata) # validating
data = obj.model_dump() # Parsing
print(data)

{'username': 'sayam_kumar', 'email': 'sayam@example.com', 'age': 25, 'is_active': True}


In [2]:
# Question 1.2: With Optional Fields
# Create a model for this data where phone and bio are optional:

pythondata = {
    "name": "Sayam",
    "email": "sayam@example.com",
    "phone": "+1-123-456-7890",
    "bio": None
}

In [8]:
from pydantic import BaseModel, Field
from typing import Optional

class DataModel(BaseModel):
  name: str
  email: str
  phone: Optional[str] = None
  bio: Optional[str] = None


obj1 = DataModel.model_validate(pythondata)


In [9]:
obj1.model_dump()

{'name': 'Sayam',
 'email': 'sayam@example.com',
 'phone': '+1-123-456-7890',
 'bio': None}

In [11]:

# Question 1.3: With Default Values
# Create a model where:

# role defaults to "user"
# login_count defaults to 0
# tags defaults to empty list

# Issue: Optional means "can be None". But defaults are actual values, not None.

pythondata = {
    "user_id": "u001",
    "name": "Sayam"
    # role, login_count, tags not provided - should use defaults
}

In [13]:
class DataModel_2(BaseModel):
  user_id: str
  name: str
  role: str = 'User'
  login_count: int = 0
  tags: list = []


obj1 = DataModel_2.model_validate(pythondata)

In [17]:
# Question 2.1: Simple Nesting
# Create models for this data:
pythondata = {
    "order_id": "ord_001",
    "customer": {
        "name": "Sayam",
        "email": "sayam@example.com"
    },
    "total_amount": 150.50
}

In [16]:
class Customer(BaseModel):
  name: str
  email: str

class DataModel(BaseModel):
  order_id: str
  customer: Customer
  total_amount: float


obj1 = DataModel.model_validate(pythondata)

In [20]:
# Question 2.2: List of Nested Models
# Create models for this data:

pythondata = {
    "playlist_id": "pl_001",
    "name": "My Favorites",
    "songs": [
        {"song_id": "s1", "title": "Song One", "duration_sec": 180},
        {"song_id": "s2", "title": "Song Two", "duration_sec": 240},
        {"song_id": "s3", "title": "Song Three", "duration_sec": 200}
    ]
}

In [21]:
from typing import List

class Song(BaseModel):
  song_id: str
  title: str
  duration_sec: int

class DataModel(BaseModel):
  playlist_id: str
  name: str
  songs: List[Song]

obj1 = DataModel.model_validate(pythondata)

In [24]:
# Question 2.3: Deeper Nesting (3 levels)
# Create models for this data:
pythondata = {
    "company_id": "comp_001",
    "name": "TechCorp",
    "departments": [
        {
            "dept_id": "d1",
            "dept_name": "Engineering",
            "employees": [
                {"emp_id": "e1", "name": "Alice", "role": "Engineer"},
                {"emp_id": "e2", "name": "Bob", "role": "Manager"}
            ]
        },
        {
            "dept_id": "d2",
            "dept_name": "Sales",
            "employees": [
                {"emp_id": "e3", "name": "Charlie", "role": "Sales Rep"}
            ]
        }
    ]
}

In [26]:
class Employee(BaseModel):
  emp_id: str
  name: str
  role: str

class Department(BaseModel):
  dept_id: str
  dept_name: str
  employees: List[Employee]

class DataModel(BaseModel):
  company_id: str
  name: str
  departments: List[Department]


obj1 = DataModel.model_validate(pythondata)

In [42]:
# Question 3.1: Basic Field Constraints
# Create a model for user registration with these rules:

# username: Required, 3-20 characters
# age: Required, must be >= 18
# email: Required
# password: Required, minimum 8 characters

pythondata = {
    "username": "sayam",
    "age": 25,
    "email": "sayam@test.com",
    "password": "sadddss123"
}

In [43]:
from pydantic import BaseModel, Field

class DataModel(BaseModel):
  username: str = Field(..., min_length=3, max_length=20)
  age: int = Field(..., gt=17)
  email: str
  password: str = Field(..., min_length=8)


obj1 = DataModel(**pythondata)

In [44]:
# Question 3.2: List Constraints
# Create a model for a quiz with these rules:

# quiz_id: Required
# title: Required, max 100 characters
# questions: Required, minimum 1 question, maximum 50 questions
# Each question has: question_id, text, options (list of 2-6 strings)

pythondata = {
    "quiz_id": "q001",
    "title": "Python Basics",
    "questions": [
        {
            "question_id": "q1",
            "text": "What is Python?",
            "options": ["Language", "Snake", "Both", "Neither"]
        },
        {
            "question_id": "q2",
            "text": "What is PEP8?",
            "options": ["Style Guide", "Library", "Framework"]
        }
    ]
}

In [48]:
class Question(BaseModel):
  question_id: str
  text: str
  options: List[str]

class Quiz(BaseModel):
  quiz_id: str
  title: str = Field(..., max_length=100)
  questions: List[Question] = Field(..., min_length=1, max_length=50)


obj1 = Quiz.model_validate(pythondata)

In [49]:
# Question 3.3: Field with Descriptions
# Create a model for ML model metadata with descriptions for each field:

# model_id: Unique model identifier
# model_name: Human-readable model name
# version: Model version string
# accuracy: Model version string
# training_samples: Number of training samples, must be positive
# labels: List of class labels, minimum 2 labels

pythondata = {
    "model_id": "intent_clf_v2",
    "model_name": "Intent Classifier",
    "version": "2.0.1",
    "accuracy": 0.95,
    "training_samples": 50000,
    "labels": ["music", "weather", "timer", "reminder"]
}

In [51]:
class ML_MODEL(BaseModel):
  model_id: str = Field(..., description='Unique model identifier')
  model_name: str = Field(..., description='Human-readable model name')
  version: str = Field(..., description='Model version string')
  accuracy: float  = Field(..., ge=0, le= 1,description='Model version string')
  training_samples: int = Field(...,gt=0, description='Number of training samples, must be positive')
  labels: List[str]= Field(..., min_length=2, description='List of class labels, minimum 2 labels')


obj1 = ML_MODEL.model_validate(pythondata)

In [53]:

# Question 4.1: Agent Trajectory (Similar to Interview)
# Create models for this agent execution data:

pythondata = {
    "run_id": "run_001",
    "agent_name": "ReAct",
    "input_query": "What's the weather in Paris?",
    "steps": [
        {
            "step_index": 0,
            "thought": "I need to check the weather in Paris",
            "action": {
                "name": "weather_api",
                "parameters": {"city": "Paris"}
            },
            "observation": "Paris: 15°C, cloudy"
        },
        {
            "step_index": 1,
            "thought": "I have the weather information",
            "action": {
                "name": "finish",
                "parameters": {"response": "The weather in Paris is 15°C and cloudy"}
            },
            "observation": None
        }
    ],
    "total_tokens": 150,
    "success": True,
    "latency_ms": 450
}

In [54]:
# Requirements:

# run_id: Required
# agent_name: Required, max 50 characters
# steps: List of steps, max 10 steps
# Each step has step_index (>= 0), thought, action (nested), observation (optional)
# action has name and parameters (dict)
# total_tokens: Must be positive
# latency_ms: Must be positive

In [57]:
from typing import Dict

class Action(BaseModel):
  name: str
  parameters: Dict[str, str]

class Step(BaseModel):
  step_index: int = Field(..., ge=0)
  thought: str
  action: Action
  observation: Optional[str]

class DataModel(BaseModel):
  run_id: str
  agent_name: str = Field(..., max_length=50)
  input_query: str
  steps: List[Step] = Field(..., max_length=10)
  total_tokens: int = Field(..., ge=0)
  success: bool
  latency_ms: int = Field(..., ge=0)

obj1 = DataModel.model_validate(pythondata)

In [64]:
obj1.model_dump()

{'run_id': 'run_001',
 'agent_name': 'ReAct',
 'input_query': "What's the weather in Paris?",
 'steps': [{'step_index': 0,
   'thought': 'I need to check the weather in Paris',
   'action': {'name': 'weather_api', 'parameters': {'city': 'Paris'}},
   'observation': 'Paris: 15°C, cloudy'},
  {'step_index': 1,
   'thought': 'I have the weather information',
   'action': {'name': 'finish',
    'parameters': {'response': 'The weather in Paris is 15°C and cloudy'}},
   'observation': None}],
 'total_tokens': 150,
 'success': True,
 'latency_ms': 450}

In [63]:
# Question 4.2: Flatten to DataFrame
# Using the model from 4.1, write a function to flatten the data into a DataFrame with columns:

# run_id
# step_index
# thought
# action_name
# observation
# success

In [67]:
import pandas as pd
def flatten_data(obj1):
  rows = []
  run_id = obj1.run_id
  success = obj1.success

  for step in obj1.steps:
    step_index = step.step_index
    thought = step.thought
    action_name = step.action.name
    observation = step.observation

    rows.append({
        'run_id': run_id,
        'step_index': step_index,
        'thought': thought,
        'action_name': action_name,
        'observation': observation,
        'success': success,
    })

  return pd.DataFrame(rows)

In [68]:
flatten_data(obj1)

,run_id,step_index,thought,action_name,observation,success
0,run_001,0,I need to check the weather in Paris,weather_api,"Paris: 15°C, cloudy",True
1,run_001,1,I have the weather information,finish,None,True


In [72]:
# Question 4.3: The Interview Question (Redo)
# Now redo your interview question correctly:
pythondata = {
    "assets": {
        "episodes": [
            {
                "episodeId": "ep_001",
                "steps": [
                    {
                        "episodeStepId": "step_001",
                        "instruction": "Open settings",
                        "imgUrl": "http://example.com/img1.png",
                        "imgSize": [274, 592],
                        "stepIndex": 0
                    },
                    {
                        "episodeStepId": "step_002",
                        "instruction": "Click next",
                        "imgUrl": "http://example.com/img2.png",
                        "imgSize": [274, 592],
                        "stepIndex": 1
                    }
                ]
            },
            {
                "episodeId": "ep_002",
                "steps": [
                    {
                        "episodeStepId": "step_003",
                        "instruction": "Tap menu",
                        "imgUrl": "http://example.com/img3.png",
                        "imgSize": [274, 592],
                        "stepIndex": 0
                    }
                ]
            }
        ]
    }
}
# Requirements:

# Max 5 episodes per file
# Max 20 steps per episode
# imgSize must be exactly 2 integers
# stepIndex must be >= 0
# Add descriptions to all fields


In [73]:
from pydantic import AnyUrl

class Step(BaseModel):
  episodeStepId: str = Field(..., description='episodeStepId')
  instruction: str = Field(..., description='instruction')
  imgUrl: AnyUrl = Field(..., description='imgUrl')
  imgSize: List[int] = Field(..., min_length=2, max_length=2)
  stepIndex: int = Field(..., ge=0)

class Episode(BaseModel):
  episodeId: str = Field(..., description='episodeId')
  steps: List[Step] = Field(..., max_length=20, description='steps per Episode, can be max 20')

class Assets(BaseModel):
  episodes: List[Episode] = Field(..., max_length=5, description='episodes per file, can be max 5')

class FileContent(BaseModel):
  assets: Assets

obj1 = FileContent.model_validate(pythondata)

In [75]:
from pprint import pprint
pprint(obj1.model_dump())

{'assets': {'episodes': [{'episodeId': 'ep_001',
                          'steps': [{'episodeStepId': 'step_001',
                                     'imgSize': [274, 592],
                                     'imgUrl': AnyUrl('http://example.com/img1.png'),
                                     'instruction': 'Open settings',
                                     'stepIndex': 0},
                                    {'episodeStepId': 'step_002',
                                     'imgSize': [274, 592],
                                     'imgUrl': AnyUrl('http://example.com/img2.png'),
                                     'instruction': 'Click next',
                                     'stepIndex': 1}]},
                         {'episodeId': 'ep_002',
                          'steps': [{'episodeStepId': 'step_003',
                                     'imgSize': [274, 592],
                                     'imgUrl': AnyUrl('http://example.com/img3.png'),
                     

In [82]:
# Also write a function to flatten to DataFrame with columns:

# episode_id
# step_id
# instruction
# step_index
# img_width
# img_height

def flatten_data(parsed_data):
  rows = []
  for episode in parsed_data.assets.episodes:
    episodeId = episode.episodeId
    for step in episode.steps:
      step_id = step.episodeStepId
      instruction= step.instruction
      step_index = step.stepIndex
      img_width = step.imgSize[1]
      img_height = step.imgSize[0]

      rows.append({
          'episodeId': episodeId,
          'step_id': step_id,
          'instruction': instruction,
          'step_index': step_index,
          'img_height': img_height,
          'img_width': img_width
      })


  return pd.DataFrame(rows)

In [83]:
flatten_data(obj1)

,episodeId,step_id,instruction,step_index,img_height,img_width
0,ep_001,step_001,Open settings,0,274,592
1,ep_001,step_002,Click next,1,274,592
2,ep_002,step_003,Tap menu,0,274,592


# 5 Pydantic Practice Questions (Tricky & Edge Cases)

In [232]:
# Question 1: Custom Field Validators
# Create a model for ML training configuration with these validations:
pythondata = {
    "model_name": "intent_classifier",
    "learning_rate": 0.001,
    "batch_size": 32,
    "epochs": 10,
    "labels": ["music", "weather", "timer"],
    "train_split": 0.8
}
# Requirements:

# model_name: Must be lowercase and alphanumeric with underscores only (no spaces or special chars)
# learning_rate: Must be between 0 and 1 (exclusive)
# batch_size: Must be a power of 2 (8, 16, 32, 64, etc.)
# epochs: Between 1 and 100
# labels: Minimum 2 labels, all labels must be lowercase
# train_split: Between 0.1 and 0.9

# Use @field_validator for custom validations.

In [233]:
from pydantic import field_validator, model_validator

class MachineLearning(BaseModel):
  model_name: str
  learning_rate: float = Field(..., ge=0, le=1)
  batch_size: int
  epochs: int = Field(..., ge=1, le=100)
  labels: List[str] = Field(..., min_length=2)
  train_split: float = Field(..., ge=0.1, le=0.9)


  @model_validator(mode='after')
  def validate_model(MachineLearning) -> MachineLearning:
    model_name = MachineLearning.model_name
    if not model_name.islower():
      raise TypeError('Model name should be lowercase')

    for c in model_name:
      if not(c.isalnum()):
        if c != '_':
          raise TypeError('Model name should contain alpha numeric char only and no special char expect underscores')

    x = MachineLearning.batch_size
    # Searched from internet
    if not ((x > 0) and (x & (x - 1))) == 0:
      raise ValueError('BatchSize should be multiple of 2')

    return MachineLearning

MachineLearning.model_validate(pythondata)


MachineLearning(model_name='intent_classifier', learning_rate=0.001, batch_size=32, epochs=10, labels=['music', 'weather', 'timer'], train_split=0.8)

In [234]:
# Question 2: Cross-Field Validation
# Create a model for an A/B experiment with cross-field validation:
pythondata = {
    "experiment_id": "exp_001",
    "start_date": "2025-01-15",
    "end_date": "2025-02-15",
    "control_group_size": 1000,
    "treatment_group_size": 1000,
    "min_sample_size": 500,
    "metrics": ["conversion_rate", "latency_p50"]
}
# Requirements:

# end_date must be after start_date
# Both control_group_size and treatment_group_size must be >= min_sample_size
# metrics must have at least 1 metric

# Use @model_validator for cross-field checks.

In [235]:
import pandas as pd

class ABTesting(BaseModel):
  experiment_id: str
  start_date: str
  end_date: str
  control_group_size: int
  treatment_group_size: int
  min_sample_size: int
  metrics: List[str] = Field(..., min_length=1)

  @model_validator(mode='after')
  def validate(self):
    start_date = pd.to_datetime(self.start_date)
    end_date = pd.to_datetime(self.end_date)
    control_group_size = self.control_group_size
    treatment_group_size = self.treatment_group_size
    min_sample_size = self.min_sample_size

    if start_date > end_date:
      raise ValueError('Start date should be less than end date')

    if (control_group_size < min_sample_size):
        raise ValueError('control_group_size should be more than min_sample_size')

    if (treatment_group_size < min_sample_size):
        raise ValueError('treatment_group_size should be more than min_sample_size')

    return self


In [236]:
ABTesting_obj = ABTesting.model_validate(pythondata)

In [237]:
# Question 3: Aliases + Extra Fields Config
# You're receiving data from an external API that uses camelCase. Create models that:

# Accept camelCase input
# Use snake_case internally
# Can output back to camelCase
# Reject any unexpected fields

pythonapi_response = {
    "requestId": "req_001",
    "modelVersion": "v2.1",
    "inputQuery": "play some music",
    "predictedIntent": "play_music",
    "confidenceScore": 0.95,
    "processingTimeMs": 45,
    "isFromCache": False
}

# After parsing, you should be able to:
# obj.request_id          # Access with snake_case
# obj.model_dump(by_alias=True)  # Output camelCase

In [238]:
# Do we need to harcode only ?
class DataModel(BaseModel):
    request_id: str = Field(alias="requestId")
    model_version: str = Field(alias="modelVersion")
    input_query: str = Field(alias="inputQuery")
    predicted_intent: str = Field(alias="predictedIntent")
    confidence_score: float = Field(alias="confidenceScore")
    processing_time_ms: int = Field(alias="processingTimeMs")
    is_from_cache: bool = Field(alias="isFromCache")

obj1 = DataModel.model_validate(pythonapi_response)

In [239]:
obj1.request_id

'req_001'

In [240]:
obj1.model_dump(by_alias=True)

{'requestId': 'req_001',
 'modelVersion': 'v2.1',
 'inputQuery': 'play some music',
 'predictedIntent': 'play_music',
 'confidenceScore': 0.95,
 'processingTimeMs': 45,
 'isFromCache': False}

In [241]:
# Question 4: Optional vs Required (Tricky Cases)
# Create a model that handles these 4 different field behaviors correctly:
# python# All these should be VALID inputs:

valid_1 = {
    "id": "001",
    "name": "Test",
    "email": "test@test.com",
    "phone": "+1234567890",
    "nickname": "Testy",
    "score": 85
}

valid_2 = {
    "id": "002",
    "name": "Test2",
    "email": None,
    "phone": "+1234567890"
}

valid_3 = {
    "id": "003",
    "name": "Test3",
    "email": "test3@test.com",
    "phone": None,
    "nickname": None,
    "score": None
}

# This should be INVALID:
invalid_1 = {
    "id": "004",
    "name": "Test4"

}

# Requirements:

# id, name: Required, cannot be None
# email: Required field but CAN be None
# phone: Required field but CAN be None
# nickname: Optional, defaults to None
# score: Optional, defaults to None
# Cross-validation: At least one of email or phone must be non-None

In [242]:
from typing import Optional

class DataModel(BaseModel):
  id: str
  name: str
  email: Optional[str] # Required but can be None
  phone: Optional[str]
  nickname: Optional[str] = None # Not Required and default to None
  score: Optional[int] = None

  @model_validator(mode='after')
  def validate(self):
    if (not self.email) and (not self.phone):
      raise TypeError('One from Email or phone should be present!')

    return self

obj1 = DataModel.model_validate(valid_2)

In [243]:
obj1.model_dump(exclude_none=False)

{'id': '002',
 'name': 'Test2',
 'email': None,
 'phone': '+1234567890',
 'nickname': None,
 'score': None}

In [244]:
# Question 5: Complete Interview Question (Combines Everything)
# You're building a data validation layer for an agent evaluation pipeline. Create models for:
pythondata = {
    "evaluationId": "eval_001",
    "modelVersion": "v2.3.1",
    "evaluatorEmail": "EVALUATOR@company.COM",
    "runDate": "2025-01-15",
    "dueDate": "2025-01-20",
    "testCases": [
        {
            "caseId": "tc_001",
            "inputQuery": "play jazz music",
            "expectedIntent": "play_music",
            "actualIntent": "play_music",
            "confidenceScore": 0.95,
            "latencyMs": 45,
            "isCorrect": True,
            "errorType": None
        },
        {
            "caseId": "tc_002",
            "inputQuery": "weather in paris",
            "expectedIntent": "get_weather",
            "actualIntent": "play_music",
            "confidenceScore": 0.35,
            "latencyMs": 120,
            "isCorrect": False,
            "errorType": "intent_mismatch"
        },
        {
            "caseId": "tc_003",
            "inputQuery": "",
            "expectedIntent": "unknown",
            "actualIntent": "unknown",
            "confidenceScore": 0.10,
            "latencyMs": 15,
            "isCorrect": True,
            "errorType": None
        }
    ],
    "summary": {
        "totalCases": 3,
        "correctCount": 2,
        "accuracy": 0.6667,
        "avgLatencyMs": 60.0,
        "avgConfidence": 0.4667
    }
}


In [245]:
# Field Constraints:

# evaluationId: Required
# modelVersion: Must match pattern v\d+\.\d+\.\d+ (e.g., v2.3.1)
# evaluatorEmail: Validate email format, normalize to lowercase
# confidenceScore: Between 0 and 1
# latencyMs: Must be positive
# testCases: Max 1000 test cases
# errorType: Can only be None, "intent_mismatch", "low_confidence", or "timeout"

# Cross-Field Validations:

# dueDate must be after runDate
# If isCorrect is True, errorType must be None
# If isCorrect is False, errorType must NOT be None
# summary.totalCases must equal length of testCases
# summary.correctCount must equal count of isCorrect=True in testCases

# Config:

# Use camelCase aliases (input is camelCase, internal is snake_case)
# Forbid extra fields

# Also write:

# Function to flatten test cases to DataFrame
# Function to calculate accuracy from the model (to verify summary is correct)

In [246]:
import re
from pydantic import field_validator, EmailStr

class TestCase(BaseModel):
  case_id: str = Field(alias="caseId")
  input_query: str = Field(alias="inputQuery")
  expected_intent: str = Field(alias="expectedIntent")
  actual_intent: str = Field(alias="actualIntent")
  confidence_score: float = Field(alias="confidenceScore", ge=0, le=1)
  latency_ms: int = Field(alias="latencyMs", ge=0)
  is_correct: bool = Field(alias="isCorrect")
  error_type: Optional[str] = Field(alias="errorType")

class Summary(BaseModel):
  total_cases: int = Field(alias="totalCases")
  correct_count: int = Field(alias="correctCount")
  accuracy: float = Field(alias="accuracy")
  avg_latency_ms: float = Field(alias="avgLatencyMs")
  avg_confidence: float = Field(alias="avgConfidence")

class DataModel(BaseModel):
  evaluation_id: str = Field(alias="evaluationId")
  model_version: str = Field(alias="modelVersion")
  evaluator_email: EmailStr = Field(alias="evaluatorEmail")
  run_date: str = Field(alias="runDate")
  due_date: str = Field(alias="dueDate")
  test_cases: List[TestCase] = Field(alias="testCases", max_length=1000)
  summary: Summary = Field(alias="summary")

  @model_validator(mode='after')
  def validate(self):
    pattern = r'v\d+\.\d+\.\d+'

    if not (re.match(pattern, self.model_version)):
      raise TypeError('Model version should match specific pattern')

    self.evaluator_email = self.evaluator_email.lower()

    iscorrectcount = 0
    for testcase in self.test_cases:
      if testcase.is_correct:
        iscorrectcount += 1

      if testcase.is_correct and testcase.error_type:
        raise TypeError('Error Should be None')

      if (not testcase.is_correct) and (not testcase.error_type):
        raise TypeError('Error Should Have some value')

      if testcase.error_type not in [None, "intent_mismatch", "low_confidence", "timeout"]:
        raise TypeError('Check Errortype it cannot be None or can only be among 3 fields')

    if self.due_date < self.run_date:
      raise TypeError('Error')

    if self.summary.total_cases != len(self.test_cases):
      raise TypeError('Error')

    if iscorrectcount != self.summary.correct_count:
      raise ValueError('Error')

    return self

In [247]:
obj1 = DataModel.model_validate(pythondata)

In [248]:
pprint(obj1.model_dump())

{'due_date': '2025-01-20',
 'evaluation_id': 'eval_001',
 'evaluator_email': 'evaluator@company.com',
 'model_version': 'v2.3.1',
 'run_date': '2025-01-15',
 'summary': {'accuracy': 0.6667,
             'avg_confidence': 0.4667,
             'avg_latency_ms': 60.0,
             'correct_count': 2,
             'total_cases': 3},
 'test_cases': [{'actual_intent': 'play_music',
                 'case_id': 'tc_001',
                 'confidence_score': 0.95,
                 'error_type': None,
                 'expected_intent': 'play_music',
                 'input_query': 'play jazz music',
                 'is_correct': True,
                 'latency_ms': 45},
                {'actual_intent': 'play_music',
                 'case_id': 'tc_002',
                 'confidence_score': 0.35,
                 'error_type': 'intent_mismatch',
                 'expected_intent': 'get_weather',
                 'input_query': 'weather in paris',
                 'is_correct': False,
         

In [249]:
obj1.model_dump(include='test_cases')

{'test_cases': [{'case_id': 'tc_001',
   'input_query': 'play jazz music',
   'expected_intent': 'play_music',
   'actual_intent': 'play_music',
   'confidence_score': 0.95,
   'latency_ms': 45,
   'is_correct': True,
   'error_type': None},
  {'case_id': 'tc_002',
   'input_query': 'weather in paris',
   'expected_intent': 'get_weather',
   'actual_intent': 'play_music',
   'confidence_score': 0.35,
   'latency_ms': 120,
   'is_correct': False,
   'error_type': 'intent_mismatch'},
  {'case_id': 'tc_003',
   'input_query': '',
   'expected_intent': 'unknown',
   'actual_intent': 'unknown',
   'confidence_score': 0.1,
   'latency_ms': 15,
   'is_correct': True,
   'error_type': None}]}

In [250]:
def flatten_data(data):
  rows = []
  for testcase in data.test_cases:
    rows.append({
        'case_id': testcase.case_id,
        'input_query': testcase.input_query,
        'expected_intent': testcase.expected_intent,
        'actual_intent': testcase.actual_intent,
        'confidence_score': testcase.confidence_score,
        'latency_ms': testcase.latency_ms,
        'is_correct': testcase.is_correct,
        'error_type': testcase.error_type
    })

  return pd.DataFrame(rows)

In [251]:
df = flatten_data(obj1)

In [252]:
accuracy = round((df['actual_intent'] == df['expected_intent']).mean(), 2)

In [253]:
float(accuracy)

0.67